In [ ]:
# Instalaivimas reikalingų paketų
!pip install git+https://github.com/huggingface/transformers accelerate
#!pip install git+https://github.com/huggingface/transformers
print("Instaliavimas baigtas")

In [ ]:
# Perspėjimų ignoravimas
import warnings
warnings.filterwarnings("ignore")  # Ignore all warnings

from transformers import BlipProcessor, BlipForConditionalGeneration                                       # SENOS KARTOS
#from transformers import Blip2Processor, Blip2ForConditionalGeneration                                      # NAUJOS KARTOS
from PIL import Image
import requests
import torch
import os
import re
import pandas as pd
print("Importavimas baigtas")

In [ ]:
# Modelių naudojimas

# Cuda įtraukimas
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Senos kartos:
model_name = "Salesforce/blip-image-captioning-base"
#model_name = "Salesforce/blip-image-captioning-large"
#model_name = "Salesforce/blip-vqa-base"
#model_name = "Salesforce/blip-vqa-capfilt-large"

# Naujos kartos:
#model_name = "Salesforce/blip2-opt-2.7b"                                 # ~2.7B 
#model_name = "Salesforce/blip2-opt-6.7b"                                 # ~6.7B 
#model_name =  "Salesforce/blip2-flan-t5-xl"                              # ~3B
#model_name =  "Salesforce/blip2-flan-t5-xxl"                             # ~11B-12B

processor = BlipProcessor.from_pretrained(model_name)                                                       # SENOS KARTOS
model = BlipForConditionalGeneration.from_pretrained(model_name, use_safetensors=True).to(device)           # SENOS KARTOS

#processor = Blip2Processor.from_pretrained(model_name)                                                      # NAUJOS KARTOS
#model = Blip2ForConditionalGeneration.from_pretrained(model_name, torch_dtype=torch.float16).to(device)     # NAUJOS KARTOS
print("Modelio krovimas baigtas")

In [ ]:
# Lentelės užkrovimas, generavimas ir saugojimas
df = pd.read_csv('flicker8k_lt.csv', sep=';')
import numpy as np

# Išsivalome lentelę
columns_to_clear = [
    'BLEU', 'ROGUE', 'METEOR',
    'BERTscore Precision', 'BERTscore Recall', 'BERTscore F1', 'SentenceBERT'
]
for col in columns_to_clear:
    if col in df.columns:
        df[col] = df[col].apply(lambda x: '')

image_folder = "Flickr8k"

#for index, row in df.head(10).iterrows():
for index, row in df.iterrows():
    filename = row["Image Name"]
    image_path = os.path.join(image_folder, filename)
    try:
        image = Image.open(image_path)
        inputs = processor(image, return_tensors="pt").to(device)
        #inputs = processor(images=image, text="Image caption", return_tensors="pt").to(model.device)
        output = model.generate(**inputs)
        generated_caption = processor.tokenizer.decode(output[0], skip_special_tokens=True)

        # Antraštės generavimas lietuviškai
        #inputs = processor(image, text="Aprašykite šią nuotrauką lietuviškai:", return_tensors="pt").to(device)
        #output = model.generate(**inputs)
        #generated_caption = processor.decode(output[0], skip_special_tokens=True)
        #print("Sugeneruota antraštė (lietuviškai):", generated_caption)

        # Įrašymas į failą
        df.at[index, "Generated Caption"] = generated_caption
        #print(generated_caption)

    except Exception as e:
        print(f"Error processing {filename}: {e}")
        df.at[index, "Generated Caption"] = f"Error: {e}"

# Saugojimas
df.to_csv("Results/flicker8k_en_01.csv", sep=';', index=False)
print("Generavimas baigtas")